In [0]:
bronze_path = "/Volumes/workspace/default/ecommerce_data/bronze/"
df = spark.read.format("delta").load(bronze_path)

print(f"Loaded {df.count()} rows from Bronze")
df.printSchema()

Loaded 1020000 rows from Bronze
root
 |-- user_id: integer (nullable = true)
 |-- age: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- urban_rural: string (nullable = true)
 |-- income_level: double (nullable = true)
 |-- employment_status: string (nullable = true)
 |-- education_level: string (nullable = true)
 |-- relationship_status: string (nullable = true)
 |-- has_children: integer (nullable = true)
 |-- household_size: integer (nullable = true)
 |-- occupation: string (nullable = true)
 |-- ethnicity: string (nullable = true)
 |-- language_preference: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- weekly_purchases: integer (nullable = true)
 |-- monthly_spend: double (nullable = true)
 |-- cart_abandonment_rate: double (nullable = true)
 |-- review_writing_frequency: integer (nullable = true)
 |-- average_order_value: double (nullable = true)
 |-- preferred_payment_method: string (nullable = tru

In [0]:
before = df.count()
df = df.dropDuplicates()
after = df.count()
print(f"Before: {before} rows")
print(f"After: {after} rows")
print(f"Duplicates removed: {before - after}")

Before: 1020000 rows
After: 1006176 rows
Duplicates removed: 13824


In [0]:
numeric_check_cols = [
    "age", "income_level", "household_size", "weekly_purchases",
    "monthly_spend", "cart_abandonment_rate", "average_order_value",
    "coupon_usage_frequency", "browse_to_buy_ratio", "return_frequency",
    "brand_loyalty_score", "travel_frequency", "hobby_count",
    "overall_stress_level", "sleep_quality", "physical_activity_level",
    "mental_health_score", "daily_session_time_minutes",
    "product_views_per_day", "account_age_months", "purchase_conversion_rate",
    "return_rate"
]
df.select(numeric_check_cols).describe().show()

+-------+------------------+------------------+------------------+-----------------+------------------+---------------------+-------------------+----------------------+-------------------+-----------------+-------------------+------------------+------------------+--------------------+------------------+-----------------------+-------------------+--------------------------+---------------------+------------------+------------------------+-----------------+
|summary|               age|      income_level|    household_size| weekly_purchases|     monthly_spend|cart_abandonment_rate|average_order_value|coupon_usage_frequency|browse_to_buy_ratio| return_frequency|brand_loyalty_score|  travel_frequency|       hobby_count|overall_stress_level|     sleep_quality|physical_activity_level|mental_health_score|daily_session_time_minutes|product_views_per_day|account_age_months|purchase_conversion_rate|      return_rate|
+-------+------------------+------------------+------------------+--------------

In [0]:
df.select("gender", "country", "employment_status", "education_level",
           "device_type", "preferred_payment_method").distinct().show(60, truncate=False)

+----------+---------+-----------------+----------------+-----------+------------------------+
|gender    |country  |employment_status|education_level |device_type|preferred_payment_method|
+----------+---------+-----------------+----------------+-----------+------------------------+
|Male      |France   |Unemployed       |Associate Degree|Desktop    |Bank Transfer           |
|Non-binary|Canada   |Retired          |Associate Degree|Desktop    |Apple Pay               |
|Male      |Canada   |Retired          |Master          |Desktop    |Credit Card             |
|Male      |Germany  |Student          |NULL            |desktop    |Bank Transfer           |
|Male      |UK       |Unemployed       |Master          |Mobile     |Bank Transfer           |
|Female    |China    |Student          |Master          |Mobile     |Google Pay              |
|Male      |Canada   |Employed         |High School     |Desktop    |Credit Card             |
|Female    |Australia|Employed         |High Schoo

In [0]:
from pyspark.sql import functions as F

cap_cols = [
    "monthly_spend", "average_order_value", "income_level",
    "daily_session_time_minutes", "product_views_per_day",
    "referral_count", "brand_loyalty_score"
]

caps = {c: df.approxQuantile(c, [0.99], 0.0001)[0] for c in cap_cols}
print("Accurate 99th percentile caps:", caps)

before_max = {c: df.agg(F.max(c)).first()[0] for c in cap_cols}
print("Before capping - max values:", before_max)

for c, cap_value in caps.items():
    df = df.withColumn(c, F.when(F.col(c) > cap_value, cap_value).otherwise(F.col(c)))

after_max = {c: df.agg(F.max(c)).first()[0] for c in cap_cols}
print("After capping - max values:", after_max)
print("Step 4 done: genuine extreme values capped (rows not deleted)")

Accurate 99th percentile caps: {'monthly_spend': 12000.0, 'average_order_value': 999.0, 'income_level': 198101.0, 'daily_session_time_minutes': 153.0, 'product_views_per_day': 50.0, 'referral_count': 10.0, 'brand_loyalty_score': 10.0}
Before capping - max values: {'monthly_spend': 150000.0, 'average_order_value': 999.0, 'income_level': 200000.0, 'daily_session_time_minutes': 156.0, 'product_views_per_day': 50.0, 'referral_count': 10, 'brand_loyalty_score': 10.0}
After capping - max values: {'monthly_spend': 12000.0, 'average_order_value': 999.0, 'income_level': 198101.0, 'daily_session_time_minutes': 153.0, 'product_views_per_day': 50.0, 'referral_count': 10.0, 'brand_loyalty_score': 10.0}
Step 4 done: genuine extreme values capped (rows not deleted)


In [0]:
print("Invalid age rows (before):", df.filter((F.col("age") < 10) | (F.col("age") > 100)).count())
age_median = df.filter((F.col("age") >= 10) & (F.col("age") <= 100)) \
               .selectExpr("percentile_approx(age, 0.5) as med").first()["med"]
print("Age median (valid range):", age_median)
df = df.withColumn(
    "age",
    F.when((F.col("age") < 10) | (F.col("age") > 100), F.lit(age_median)).otherwise(F.col("age"))
)

print("Invalid household_size rows (before):", df.filter((F.col("household_size") < 1) | (F.col("household_size") > 15)).count())
hh_median = df.filter((F.col("household_size") >= 1) & (F.col("household_size") <= 15)) \
              .selectExpr("percentile_approx(household_size, 0.5) as med").first()["med"]
print("Household size median (valid range):", hh_median)
df = df.withColumn(
    "household_size",
    F.when((F.col("household_size") < 1) | (F.col("household_size") > 15), F.lit(hh_median)).otherwise(F.col("household_size"))
)

print("Negative monthly_spend rows (before):", df.filter(F.col("monthly_spend") < 0).count())
spend_median = df.filter(F.col("monthly_spend") >= 0) \
                  .selectExpr("percentile_approx(monthly_spend, 0.5) as med").first()["med"]
print("Monthly spend median (valid range):", spend_median)
df = df.withColumn(
    "monthly_spend",
    F.when(F.col("monthly_spend") < 0, F.lit(spend_median)).otherwise(F.col("monthly_spend"))
)

df.select("age", "household_size", "monthly_spend").describe().show()

Invalid age rows (before): 2000
Age median (valid range): 49.0
Invalid household_size rows (before): 2000
Household size median (valid range): 6
Negative monthly_spend rows (before): 657
Monthly spend median (valid range): 2210.0
+-------+------------------+------------------+------------------+
|summary|               age|    household_size|     monthly_spend|
+-------+------------------+------------------+------------------+
|  count|            986076|           1006176|            986081|
|   mean|49.000763632823436|  5.50592441083866|3013.3370787998147|
| stddev| 18.17333914656605|2.8707721011370917| 2676.581974172611|
|    min|              18.0|                 1|               0.0|
|    max|              80.0|                10|           12000.0|
+-------+------------------+------------------+------------------+



In [0]:
df = df.withColumn("last_purchase_date_parsed", F.to_date(F.col("last_purchase_date")))

median_date = df.filter(F.col("last_purchase_date_parsed").isNotNull()) \
                 .selectExpr("percentile_approx(unix_date(last_purchase_date_parsed), 0.5) as med") \
                 .first()["med"]

median_date_str = df.select(F.date_add(F.lit("1970-01-01"), int(median_date)).alias("d")).first()["d"]
print("Median last_purchase_date:", median_date_str)

df = df.withColumn(
    "last_purchase_date",
    F.when(F.col("last_purchase_date").isNull(), F.lit(str(median_date_str))).otherwise(F.col("last_purchase_date"))
)
df = df.drop("last_purchase_date_parsed")

print("Nulls remaining in last_purchase_date:", df.filter(F.col("last_purchase_date").isNull()).count())

Median last_purchase_date: 2026-01-02
Nulls remaining in last_purchase_date: 0


In [0]:
country_map = {
    "austrailia": "Australia", "australia": "Australia",
    "brasil": "Brazil", "brazil": "Brazil",
    "canada": "Canada", "canda": "Canada",
    "china": "China", "chinaa": "China",
    "franc": "France", "france": "France",
    "germany": "Germany", "germny": "Germany",
    "india": "India", "inida": "India",
    "japan": "Japan", "jappan": "Japan",
    "uk": "UK", "u.k.": "UK",
    "usa": "USA", "u.s.a": "USA",
}
map_country = F.create_map([F.lit(x) for pair in country_map.items() for x in pair])
df = df.withColumn("country", map_country[F.lower(F.trim(F.col("country")))])

category_map = {
    "beauty": "Beauty", "beuty": "Beauty",
    "book": "Books", "books": "Books",
    "electroincs": "Electronics", "electronic": "Electronics", "electronics": "Electronics",
    "fashion": "Fashion", "fasion": "Fashion",
    "groceries": "Groceries", "grocery": "Groceries",
    "home & kitchen": "Home & Kitchen", "home and kitchen": "Home & Kitchen",
    "sport": "Sports", "sports": "Sports",
    "toy": "Toys", "toys": "Toys",
}
map_category = F.create_map([F.lit(x) for pair in category_map.items() for x in pair])
df = df.withColumn(
    "product_category_preference",
    map_category[F.lower(F.trim(F.col("product_category_preference")))]
)

gender_map = {
    "male": "Male", "mael": "Male",
    "female": "Female",
    "non-binary": "Non-binary",
    "other": "Other",
}
map_gender = F.create_map([F.lit(x) for pair in gender_map.items() for x in pair])
df = df.withColumn("gender", map_gender[F.lower(F.trim(F.col("gender")))])

device_map = {
    "desktop": "Desktop",
    "mobile": "Mobile",
    "tablet": "Tablet",
}
map_device = F.create_map([F.lit(x) for pair in device_map.items() for x in pair])
df = df.withColumn("device_type", map_device[F.lower(F.trim(F.col("device_type")))])

payment_map = {
    "credit card": "Credit Card",
    "debit card": "Debit Card",
    "paypal": "PayPal",
    "apple pay": "Apple Pay",
    "google pay": "Google Pay",
    "bank transfer": "Bank Transfer",
}
map_payment = F.create_map([F.lit(x) for pair in payment_map.items() for x in pair])
df = df.withColumn("preferred_payment_method", map_payment[F.lower(F.trim(F.col("preferred_payment_method")))])

df.select("country").distinct().orderBy("country").show(20, truncate=False)
df.select("product_category_preference").distinct().orderBy("product_category_preference").show(20, truncate=False)
df.select("gender").distinct().orderBy("gender").show()
df.select("device_type").distinct().orderBy("device_type").show()
df.select("preferred_payment_method").distinct().orderBy("preferred_payment_method").show()

+---------+
|country  |
+---------+
|Australia|
|Brazil   |
|Canada   |
|China    |
|France   |
|Germany  |
|India    |
|Japan    |
|UK       |
|USA      |
+---------+

+---------------------------+
|product_category_preference|
+---------------------------+
|NULL                       |
|Beauty                     |
|Books                      |
|Electronics                |
|Fashion                    |
|Groceries                  |
|Home & Kitchen             |
|Sports                     |
|Toys                       |
+---------------------------+

+----------+
|    gender|
+----------+
|      NULL|
|    Female|
|      Male|
|Non-binary|
|     Other|
+----------+

+-----------+
|device_type|
+-----------+
|       NULL|
|    Desktop|
|     Mobile|
|     Tablet|
+-----------+

+------------------------+
|preferred_payment_method|
+------------------------+
|                    NULL|
|               Apple Pay|
|           Bank Transfer|
|             Credit Card|
|              Debit

In [0]:
fill_unknown_cols = [
    "employment_status", "education_level", "product_category_preference",
    "device_type", "preferred_payment_method", "gender", "country"
]

for c in fill_unknown_cols:
    before_nulls = df.filter(F.col(c).isNull()).count()
    df = df.withColumn(c, F.when(F.col(c).isNull(), F.lit("Unknown")).otherwise(F.col(c)))
    print(f"{c}: {before_nulls} nulls -> filled with Unknown")

employment_status: 20119 nulls -> filled with Unknown
education_level: 20144 nulls -> filled with Unknown
product_category_preference: 20113 nulls -> filled with Unknown
device_type: 21524 nulls -> filled with Unknown
preferred_payment_method: 32379 nulls -> filled with Unknown
gender: 25263 nulls -> filled with Unknown
country: 0 nulls -> filled with Unknown


In [0]:
print("Total rows:", df.count())
print("Total columns:", len(df.columns))
null_counts = df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns])
null_counts.show(vertical=True, truncate=False)

Total rows: 1006176
Total columns: 60
-RECORD 0--------------------------------
 user_id                         | 0     
 age                             | 20100 
 gender                          | 0     
 country                         | 0     
 urban_rural                     | 0     
 income_level                    | 20117 
 employment_status               | 0     
 education_level                 | 0     
 relationship_status             | 0     
 has_children                    | 0     
 household_size                  | 0     
 occupation                      | 0     
 ethnicity                       | 0     
 language_preference             | 0     
 device_type                     | 0     
 weekly_purchases                | 0     
 monthly_spend                   | 20095 
 cart_abandonment_rate           | 20105 
 review_writing_frequency        | 0     
 average_order_value             | 20132 
 preferred_payment_method        | 0     
 coupon_usage_frequency          | 201

In [0]:
dbutils.fs.rm("/Volumes/workspace/default/ecommerce_data/silver/", recurse=True)
print("Old silver path cleared")

silver_path = "/Volumes/workspace/default/ecommerce_data/silver/"

df.write.format("delta").mode("overwrite").save(silver_path)

print(f"Silver table written: {df.count()} rows, {len(df.columns)} columns")
print(f"Path: {silver_path}")

Old silver path cleared
Silver table written: 1006176 rows, 60 columns
Path: /Volumes/workspace/default/ecommerce_data/silver/
